# Three-Term Decomposition: T1 / T2 / T3

This notebook presents the three-term evaluation of gradient attribution on Boltz-2:

- **T1**: attribution vs the model's own mutation scan (is the gradient faithful?)
- **T2**: the model's scan vs experiment (is the model right about biology?)
- **T3**: attribution vs experiment (what a practitioner actually gets)

T3 is compared against published AbBiBench baselines.

In [ ]:
import sys
from pathlib import Path

METRICS_PATH = Path("../results/metrics.csv")

if not METRICS_PATH.exists():
    print(f"Missing: {METRICS_PATH}")
    print("Run: python scripts/06_metrics.py --pred <prediction_csv>")
    sys.exit(1)

print("Metrics file found.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

metrics = pd.read_csv(METRICS_PATH)
print(f"Metrics table: {len(metrics)} rows")
print(metrics.head(20))

## T1 / T2 / T3 side by side

In [ ]:
PUBLISHED_BASELINES = {
    "ProteinMPNN": 0.30,
    "ESM-IF1": 0.28,
    "AntiFold": 0.21,
    "Boltz-2_scan": 0.13,
    "FoldX": 0.12,
    "AF3": -0.02,
}

term_rows = metrics[metrics["score"] == "spearman"]
our_terms = term_rows[term_rows["term"].isin(["T1", "T2", "T3"])]

if len(our_terms) > 0:
    fig, ax = plt.subplots(figsize=(6, 4))
    terms = our_terms.set_index("term")["value"]
    colors = {"T1": "#2196F3", "T2": "#FF9800", "T3": "#4CAF50"}
    for t in ["T1", "T2", "T3"]:
        if t in terms.index:
            ax.bar(t, terms[t], color=colors[t], width=0.5)
            ax.text(t, terms[t] + 0.005, f"{terms[t]:.3f}", ha="center", va="bottom", fontsize=10)
    ax.set_ylabel("Spearman correlation")
    ax.set_title("Three-term decomposition")
    ax.axhline(0, color="gray", linewidth=0.5)
    plt.tight_layout()
    plt.show()
else:
    print("No T1/T2/T3 Spearman rows found in metrics. Only T3 is guaranteed; T1/T2 require --scan.")

## T3 vs published baselines

In [ ]:
t3_row = term_rows[term_rows["term"] == "T3"]
t3_val = t3_row["value"].values[0] if len(t3_row) > 0 else None

labels = list(PUBLISHED_BASELINES.keys()) + ["Ours (T3)"]
values = list(PUBLISHED_BASELINES.values()) + [t3_val if t3_val is not None else 0.0]

sorted_pairs = sorted(zip(values, labels), reverse=True)
values_sorted = [v for v, _ in sorted_pairs]
labels_sorted = [l for _, l in sorted_pairs]

colors = ["#4CAF50" if l == "Ours (T3)" else "#90CAF9" for l in labels_sorted]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(range(len(labels_sorted)), values_sorted, color=colors)
ax.set_yticks(range(len(labels_sorted)))
ax.set_yticklabels(labels_sorted)
ax.set_xlabel("Spearman correlation (per-dataset, averaged)")
ax.set_title("Attribution vs published baselines")
ax.axvline(0.13, color="red", linestyle="--", linewidth=1, label="Boltz-2 scan (0.13)")
ax.legend(loc="lower right")

for i, v in enumerate(values_sorted):
    ax.text(v + 0.005 if v >= 0 else v - 0.02, i, f"{v:.2f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

## Interpretation

Read the numbers from the metrics table and state the conclusions plainly.

In [ ]:
boltz_scan = PUBLISHED_BASELINES["Boltz-2_scan"]

if t3_val is not None:
    beat_scan = t3_val > boltz_scan
    best_baseline_name = max(PUBLISHED_BASELINES, key=PUBLISHED_BASELINES.get)
    best_baseline_val = PUBLISHED_BASELINES[best_baseline_name]
    beat_best = t3_val > best_baseline_val

    print(f"T3 (attribution vs experiment): {t3_val:.4f}")
    print(f"Boltz-2 brute-force scan:       {boltz_scan:.2f}")
    print()
    if beat_scan:
        print(f"Attribution BEATS the brute-force scan ({t3_val:.3f} > {boltz_scan:.2f}).")
    else:
        print(f"Attribution does NOT beat the brute-force scan ({t3_val:.3f} <= {boltz_scan:.2f}).")
    print()
    if beat_best:
        print(f"Attribution BEATS all published baselines (best: {best_baseline_name} = {best_baseline_val:.2f}).")
    else:
        print(f"Attribution does NOT beat the best baseline ({best_baseline_name} = {best_baseline_val:.2f}).")
else:
    print("T3 value not available. Run 06_metrics.py first.")

## Conclusion

The three-term decomposition separates the question of whether gradient attribution
works into: (T1) is the gradient a faithful proxy for the model's own predictions,
(T2) does the model predict biology correctly, and (T3) what does the practitioner
actually get. The bar chart above places our T3 against the published AbBiBench
leaderboard, with the Boltz-2 brute-force scan (0.13) as the primary comparison.
If T3 exceeds 0.13, attribution successfully replaces the expensive mutation scan
with a single backward pass.